# SHAP Deep Dive — Inflow vs Outflow para C102290

**Pergunta:** O `log_pix_in` realmente tem mais peso que o outflow, ou o outflow está fragmentado/escondido?

**Metodologia:**
1. Replicar exatamente o pipeline do `ml_pipeline.py`
2. Extrair os SHAP values brutos do C102290 (não apenas o top-3)
3. Agrupar features em **INFLOW**, **OUTFLOW**, **INCOME MISMATCH**, **ANONIMIZAÇÃO**, e **OUTROS**
4. Comparar as contribuições agregadas por grupo
5. Waterfall plot completo com todos os drivers


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent          # notebooks/ -> repo root
sys.path.insert(0, str(ROOT / "src" / "rules"))
sys.path.insert(0, str(ROOT / "src" / "ml"))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xgboost as xgb
import shap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.model_selection import train_test_split

import alerts_engine as ae
import ml_pipeline as mp

%matplotlib inline
print("Setup OK")


## 1. Carregar dados e treinar modelo (idêntico ao ml_pipeline.py)

In [ ]:
txs, kyc, mer = ae.load_data()
txs["timestamp"] = pd.to_datetime(txs["timestamp"])

txs_window = txs[(txs.timestamp >= mp.FEAT_START) & (txs.timestamp < mp.FEAT_END)]

X_all = mp.build_features(txs_window, kyc, mer)
lbl   = mp.build_labels(txs_window, kyc, mer)

labeled = lbl.dropna(subset=["y"]).index
X = X_all.loc[labeled].astype(float)
y = lbl.loc[labeled, "y"].astype(int)

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42
)

print("Treinando XGBoost...")
model = mp.train_xgb(X_dev, y_dev)
print("Modelo treinado.")

X_full = X_all.astype(float)
explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X_full)
probs_full = model.predict_proba(X_full)[:, 1]
feat_names = X_full.columns.tolist()

print(f"SHAP calculado para {len(X_full)} clientes.")
print(f"Features: {len(feat_names)}")


## 2. SHAP values brutos do C102290 — todos os features

In [ ]:
SUBJECT = "C102290"

idx = X_full.index.get_loc(SUBJECT)
sv  = shap_vals[idx]          # array com 1 valor por feature
fv  = X_full.iloc[idx]        # feature values do cliente

prob = float(probs_full[idx])
print(f"C102290 — predicted probability: {prob:.4f}")
print(f"Expected value (base rate):       {explainer.expected_value:.4f}")
print(f"Sum of SHAP values:               {sv.sum():.4f}  (prob em log-odds space, não linear)")

# DataFrame completo: feature, shap value, feature value original
shap_df = pd.DataFrame({
    "feature":     feat_names,
    "shap_value":  sv,
    "feat_value":  fv.values,
}).sort_values("shap_value", ascending=False)

# Separar positivos (empurram pra suspeito) e negativos (atenuam)
pos_shap = shap_df[shap_df.shap_value > 0]
neg_shap = shap_df[shap_df.shap_value < 0]

print(f"\nFeatures com contribuição POSITIVA (aumentam suspeita): {len(pos_shap)}")
print(f"Features com contribuição NEGATIVA (atenuam suspeita):  {len(neg_shap)}")
print()
print(shap_df.to_string(index=False))


## 3. Agrupamento: INFLOW vs OUTFLOW vs demais grupos

In [ ]:
# ── Classificação manual de cada feature em grupos ────────────────────────────
# Baseado na lógica do build_features() em ml_pipeline.py

GROUPS = {
    "INFLOW": [
        "log_pix_in",
    ],
    "OUTFLOW": [
        "log_total_outflow",
        "log_pix_outflow",   # rail column: log_pix_outflow
        "log_pix_out",
        "log_wire_outflow",
        "log_card_outflow",
        "log_pix_outflow",
        "count_wire_sent",
    ],
    "INCOME_MISMATCH": [
        "log_annual_income",
        "kyc_risk_score",
        "risk_rating_ord",
        "kyc_tier_ord",
        "pep_flag",
    ],
    "ANONIMIZAÇÃO": [
        "count_anon_events",
        "count_tor_events",
        "count_rooted_tx",
        "distinct_devices",
        "distinct_ip_addresses",
        "count_ip_mismatch",
    ],
    "COMPORTAMENTAL": [
        "tx_count",
        "distinct_active_days",
        "max_txs_per_day",
        "mean_tx_amount",
        "std_tx_amount",
        "max_tx_amount",
        "fraction_night_activity",
        "fraction_weekend_activity",
    ],
    "GEO": [
        "distinct_geo_countries",
        "count_high_risk_geo",
        "fraction_cross_border",
    ],
    "MERCHANT": [
        "distinct_merchants",
        "mean_merchant_chargeback",
        "fraction_high_mcc",
        "distinct_mccs",
    ],
    "REDE": [
        "count_c2c_tx",
        "distinct_c2c_counterparties",
    ],
    "CARD": [
        "count_cnp",
        "count_no_3ds",
        "count_chargeback_status",
    ],
    "KYC_DEMO": [
        "age",
    ],
}

# Mapeamento feature -> grupo (com fallback "OUTROS")
feat_to_group = {}
for grp, feats in GROUPS.items():
    for f in feats:
        feat_to_group[f] = grp

shap_df["group"] = shap_df["feature"].map(feat_to_group).fillna("OUTROS")

# Checar features não mapeadas
unmapped = shap_df[shap_df["group"] == "OUTROS"]["feature"].tolist()
if unmapped:
    print("Features não mapeadas explicitamente (vão para OUTROS):")
    for f in unmapped:
        print(f"  {f}")
else:
    print("Todas as features mapeadas com sucesso.")

# Soma de SHAP por grupo
group_shap = (shap_df.groupby("group")["shap_value"]
                     .sum()
                     .sort_values(ascending=False)
                     .reset_index())
group_shap.columns = ["grupo", "shap_total"]

print()
print("=" * 50)
print("SHAP TOTAL POR GRUPO — C102290")
print("=" * 50)
for _, row in group_shap.iterrows():
    bar = "█" * int(abs(row.shap_total) * 8)
    sinal = "+" if row.shap_total > 0 else "-"
    print(f"  {row.grupo:20s}  {sinal}{abs(row.shap_total):.4f}  {bar}")


## 4. A questão central: INFLOW vs OUTFLOW (incluindo income mismatch)

O argumento é:
- `log_pix_in` aparece como top driver → **INFLOW**
- `log_annual_income` captura o gap renda vs movimentação → **parte do OUTFLOW indiretamente**
- `log_wire_outflow` + `log_pix_out` estão fragmentados

Vamos somar os grupos que representam "o dinheiro que sai é absurdo":


In [ ]:
# Contribuição isolada de cada feature de inflow/outflow
print("=" * 65)
print("CONTRIBUIÇÃO INDIVIDUAL — features de fluxo financeiro")
print("=" * 65)

flow_features = [
    "log_pix_in",
    "log_pix_out", "log_pix_outflow",
    "log_wire_outflow",
    "log_card_outflow",
    "log_total_outflow",
    "log_annual_income",
    "count_wire_sent",
]

flow_df = shap_df[shap_df["feature"].isin(flow_features)].copy()
flow_df = flow_df.sort_values("shap_value", ascending=False)

for _, row in flow_df.iterrows():
    direction = "← INFLOW" if "pix_in" in row.feature else (
                "← INCOME (gap)" if "income" in row.feature else "← OUTFLOW")
    print(f"  {row.feature:28s}  shap={row.shap_value:+.4f}  val={row.feat_value:>10.3f}  {direction}")

print()
# ── A soma que importa ────────────────────────────────────────────────────────
inflow_total  = shap_df[shap_df["feature"] == "log_pix_in"]["shap_value"].sum()

outflow_feats = ["log_pix_out", "log_pix_outflow", "log_wire_outflow",
                 "log_card_outflow", "log_total_outflow", "count_wire_sent"]
outflow_total = shap_df[shap_df["feature"].isin(outflow_feats)]["shap_value"].sum()

# income mismatch: renda baixa AUMENTA suspeita (shap positivo quando renda é baixa)
income_shap   = shap_df[shap_df["feature"] == "log_annual_income"]["shap_value"].sum()

print("=" * 65)
print("RESUMO AGREGADO")
print("=" * 65)
print(f"  INFLOW  (log_pix_in)                   : {inflow_total:+.4f}")
print(f"  OUTFLOW (todos os outflow features)     : {outflow_total:+.4f}")
print(f"  INCOME MISMATCH (log_annual_income)     : {income_shap:+.4f}")
print(f"  OUTFLOW + INCOME (gap renda vs saída)   : {outflow_total + income_shap:+.4f}")
print()
print(f"  → INFLOW puro:                {inflow_total:+.4f}")
print(f"  → OUTFLOW+MISMATCH combinado: {outflow_total + income_shap:+.4f}")

if abs(outflow_total + income_shap) > abs(inflow_total):
    print("\n  ✓ CONFIRMADO: Outflow+mismatch > Inflow quando somados corretamente.")
else:
    print("\n  → Inflow ainda domina mesmo na soma combinada.")
    diff = abs(inflow_total) - abs(outflow_total + income_shap)
    print(f"     Diferença: {diff:.4f}")


## 5. Waterfall plot — todos os drivers do C102290

In [ ]:
# Waterfall plot manual (top 15 features por |shap|)
top_n = 15
top_df = shap_df.reindex(shap_df["shap_value"].abs().sort_values(ascending=False).index).head(top_n)
top_df = top_df.sort_values("shap_value")  # crescente para plotar horizontal

colors = ["#d62728" if v > 0 else "#1f77b4" for v in top_df["shap_value"]]

fig, ax = plt.subplots(figsize=(10, 7))
bars = ax.barh(top_df["feature"], top_df["shap_value"], color=colors, edgecolor="white", height=0.7)

ax.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars, top_df.iterrows()):
    x = bar.get_width()
    label = f"{x:+.3f}  (val={row.feat_value:.2f})"
    ax.text(
        x + (0.002 if x >= 0 else -0.002),
        bar.get_y() + bar.get_height() / 2,
        label,
        va="center",
        ha="left" if x >= 0 else "right",
        fontsize=8,
        color="black"
    )

ax.set_xlabel("SHAP value (contribuição para log-odds de suspeita)", fontsize=10)
ax.set_title(
    f"C102290 — Top {top_n} SHAP drivers\n"
    f"prob={prob:.4f}  |  base rate={explainer.expected_value:.4f}",
    fontsize=11
)

red_patch  = mpatches.Patch(color="#d62728", label="Aumenta suspeita (→ positivo)")
blue_patch = mpatches.Patch(color="#1f77b4", label="Atenua suspeita (→ negativo)")
ax.legend(handles=[red_patch, blue_patch], fontsize=9, loc="lower right")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("shap_waterfall_C102290.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_waterfall_C102290.png")


## 6. SHAP por grupo — visão consolidada

In [ ]:
group_colors = {
    "INFLOW":          "#e74c3c",
    "OUTFLOW":         "#e67e22",
    "INCOME_MISMATCH": "#f39c12",
    "ANONIMIZAÇÃO":    "#8e44ad",
    "COMPORTAMENTAL":  "#2980b9",
    "GEO":             "#27ae60",
    "MERCHANT":        "#16a085",
    "REDE":            "#2c3e50",
    "CARD":            "#7f8c8d",
    "KYC_DEMO":        "#bdc3c7",
    "OUTROS":          "#ecf0f1",
}

gs = group_shap.copy()
gs["color"] = gs["grupo"].map(group_colors).fillna("#cccccc")
gs = gs.sort_values("shap_total")

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(gs["grupo"], gs["shap_total"], color=gs["color"], edgecolor="white", height=0.65)

ax.axvline(0, color="black", linewidth=0.8)

for bar, (_, row) in zip(bars, gs.iterrows()):
    x = bar.get_width()
    ax.text(
        x + (0.001 if x >= 0 else -0.001),
        bar.get_y() + bar.get_height() / 2,
        f"{x:+.4f}",
        va="center",
        ha="left" if x >= 0 else "right",
        fontsize=9
    )

ax.set_xlabel("SHAP total por grupo", fontsize=10)
ax.set_title(
    "C102290 — Contribuição SHAP agregada por grupo de features\n"
    "(positivo = aumenta suspeita, negativo = atenua)",
    fontsize=11
)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("shap_groups_C102290.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvo: shap_groups_C102290.png")


## 7. Conclusão

In [ ]:
print("=" * 65)
print("VEREDICTO FINAL — C102290")
print("=" * 65)
print()

total_positive = shap_df[shap_df.shap_value > 0]["shap_value"].sum()
total_negative = shap_df[shap_df.shap_value < 0]["shap_value"].sum()

inflow_pct  = inflow_total  / total_positive * 100
outflow_pct = outflow_total / total_positive * 100 if outflow_total > 0 else 0
income_pct  = income_shap   / total_positive * 100 if income_shap  > 0 else 0

print(f"  Soma de TODAS as contribuições positivas: {total_positive:+.4f}")
print(f"  Soma de TODAS as contribuições negativas: {total_negative:+.4f}")
print()
print(f"  INFLOW  representa {inflow_pct:.1f}% das contribuições positivas")
print(f"  OUTFLOW representa {outflow_pct:.1f}% das contribuições positivas")
print(f"  INCOME  representa {income_pct:.1f}% das contribuições positivas")
print()

outflow_combined_pct = (outflow_total + income_shap) / total_positive * 100
print(f"  OUTFLOW + INCOME MISMATCH combinados: {outflow_combined_pct:.1f}%")
print()

if outflow_combined_pct > inflow_pct:
    print("  ✓ A hipótese foi CONFIRMADA:")
    print("    O outflow, quando somado com o income mismatch, supera o inflow.")
    print("    O SHAP estava correto — a fragmentação escondia o peso real do outflow.")
else:
    print("  → O inflow ainda domina mesmo na soma combinada.")
    print("    O modelo genuinamente identifica o INFLOW anômalo como sinal primário.")
    print("    O outflow é consequência esperada — menos discriminante na população.")
print()
print("  Em ambos os casos: o SHAP está correto. A questão era de interpretação,")
print("  não de erro no modelo.")
